In [0]:
source_table = dbutils.widgets.get("source_table")
staging_table = dbutils.widgets.get("staging_table")
source_system_table = dbutils.widgets.get("source_system_table")
office_table = dbutils.widgets.get("office_table")
payer_table = dbutils.widgets.get("payer_table")
weekending_table = dbutils.widgets.get("weekending_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
%skip
source_table = "dev_silver_lakehouse.staging.bears_oblist"
staging_table = "dev_silver_lakehouse.staging.bears_oblist_stg"
source_system_table = "prd_bronze_raw.mart_bkp.cubeserviceofficetxnsourcesystem"
office_table = "prd_bronze_raw.mart_bkp.office"
payer_table = "prd_bronze_raw.staging_bkp.payerdimension"
weekending_table = "prd_bronze_raw.mart_bkp.cubeserviceofficetxnweekendingdate"
client_table = "prd_bronze_raw.mart_bkp.client"

In [0]:
spark.sql(f"""
TRUNCATE TABLE {staging_table};
""")

In [0]:
spark.sql(f"""
INSERT INTO {staging_table}        
SELECT 
  S.SourceSystemKey AS source_system_key,
  W.WeekEndingDateKey AS reporting_week_ending_date_key,
  O.OfficeKey AS office_key,
  C.ClientKey AS client_key,
  OL.client_no AS client_no,
  OL.bill_to AS bill_to,
  P.PayerKey AS payer_key,
  OL.payor_type_code AS payor_type_code,
  OL.collector_name AS collector_name,
  OL.inv_no AS inv_no,
  OL.ar_0_90 AS ar_0_90,
  OL.ar_91_180 AS ar_91_180,
  OL.ar_181_270 AS ar_181_270,
  OL.ar_271_plus AS ar_271_plus
FROM
(
    SELECT 
        'BEARS' AS source_system,
        CAST(Ob.reporting_week_ending_date AS TIMESTAMP) AS reporting_period,
        Ob.office,
        Ob.client_no,
        Ob.client_name,
        Ob.payor_name,
        Ob.bill_to,
        Ob.payor_type_code,
        Ob.collector_name,
        Ob.inv_no,
        SUM(COALESCE(Ob.current_amt, 0)) + SUM(COALESCE(Ob.weeks_4_7, 0)) + SUM(COALESCE(Ob.weeks_8_13, 0)) AS ar_0_90,
        SUM(COALESCE(Ob.weeks_14_to_reserve, 0)) AS ar_91_180,
        SUM(COALESCE(Ob.up_for_reserve, 0)) AS ar_181_270,
        SUM(COALESCE(Ob.prior_reserve, 0)) AS ar_271_plus
    FROM {source_table} Ob
    GROUP BY 
        Ob.office,
        Ob.client_no,
        Ob.client_name,
        Ob.inv_no,
        Ob.payor_name,
        Ob.payor_type_code,
        Ob.collector_name,
        Ob.reporting_week_ending_date,
        Ob.bill_to
) OL
LEFT JOIN {source_system_table} S  
  ON S.SourceSystemName = OL.source_system
LEFT JOIN {office_table} O 
  ON O.OfficeNumber = OL.office
LEFT JOIN {payer_table} P    
  ON P.PayerID = OL.bill_to
LEFT JOIN {weekending_table} W  
  ON W.WeekEndingDate = OL.reporting_period
LEFT JOIN {client_table} C 
  ON C.SourceSystemId = OL.client_no 
  AND C.OfficeNumber = OL.office
""")